![DB Academy](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/common/db-academy.png)

# 05 - Deploying a Simple Declarative Automation Bundle (DAB)

## Overview

In this demonstration, you'll create a simple Databricks job, examine its YAML configuration, and walk through the complete lifecycle of a **Declarative Automation Bundle (DAB)**: validate, deploy, run, modify, redeploy, and destroy. Everything runs from a Databricks notebook for training convenience. The Databricks CLI install and authentication were handled in **02 - REQUIRED - Course Setup and Authentication**.

**Reference documentation:**
- **What are Databricks Asset Bundles / Declarative Automation Bundles?**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/)
- **Bundle configuration reference (YAML)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/reference) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/reference) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/reference)
- **`databricks bundle` CLI commands**: [AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

## Learning Objectives

By the end of this demonstration, you will be able to:

1. **Explain the purpose** of the **databricks.yml** configuration file in a DAB.
2. **Identify the key sections** (`bundle`, `resources`, `targets`, `workspace`) in a bundle configuration.
3. **Generate a YAML job configuration** from an existing job using **View as code**.
4. **Validate, deploy, and run a job** using `databricks bundle validate`, `databricks bundle deploy`, and `databricks bundle run`.
5. **Modify and redeploy** a bundle to update an existing job.
6. **Destroy a bundle** with `databricks bundle destroy` to clean up deployed resources.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>



## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Recall that your environment was setup using the **02 - REQUIRED - Course Setup and Authentication**.

If you end your lab or your lab session times out, your environment will be reset. You will need to rerun the **02 - REQUIRED - Course Setup and Authentication** notebook to recreate the catalogs and data for your environment.

  </div>
</div>



## A. Classroom Setup

Run the following cell to configure your working environment for this course.

In [0]:
%run ../Includes/Classroom-Setup-05

## B. Create and Explore a Simple Job

### B1. Create a Lakeflow Job

1. During development, it's easier to manually create the job you want to automatically deploy with Declarative Automation Bundles in order to get the necessary YAML configuration for deployment.

    Run the cell below and confirm that the job was created.

    **NOTE:** To save time, we will use the Databricks Academy `DAJobConfig` class, which was created using the Databricks SDK to automatically create our job for this demonstration. In a typical development cycle, you would create the job manually.

In [0]:
job_tasks = [
    {
        'task_name': 'create_bronze_table',
        'notebook_path': '/05 - Deploying a Simple DAB/src/create_bronze_table',
        'depends_on': None
    },
    {
        'task_name': 'create_silver_table',
        'notebook_path': '/05 - Deploying a Simple DAB/src/create_silver_table',
        'depends_on': [{'task_key': 'create_bronze_table'}]
    }
]

myjob = DAJobConfig(job_name=f'demo05_simple_dab_{my_catalog}',
                    job_tasks=job_tasks,
                    job_parameters=[
                      {'name':'display_target', 'default':'development'},
                      {'name':'catalog_name', 'default':catalog_dev}
                    ])

### B2. Explore the Job Configurations
Complete the following steps to explore the YAML configuration of the job:

1. In the left main navigation bar, right-click on **Jobs and Pipelines** and select **Open in a new tab**.

2. Locate your deployed job named **demo05_simple_dab_LABUSER_UNIQUE_ID**.

3. Select your job.

4. In the right **Job details** pane, scroll to the bottom and find **Job parameters**. Notice that two parameters have been set for this job:
   | Job parameters | Description |
   |---|---|
   | `catalog_name` | References your **labuser_UNIQUE_ID** catalog. |
   | `display_target` | Text value that specifies the environment where the job is running. In this example, we are using `development`. |

5. In the top navigation bar, select **Tasks**. Notice that this job has two tasks:
| Task | Description |
|---|---|
| **TASK 1** | - Runs the notebook **create_bronze_table**<br>- Reads from the development CSV file in the **labuser_UNIQUE_ID_dev** catalog<br>- Uses the job parameter `catalog_name`<br>- Creates the table **health_bronze_demo_05** |
| **TASK 2** | - Runs the notebook **create_silver_table**<br>- Reads from the bronze table in the **labuser_UNIQUE_ID_dev** catalog<br>- Uses the job parameter `catalog_name`<br>- Creates the table **health_silver_demo_05**<br>- Depends on **TASK 1** completing successfully |
    
    - Both tasks use **Serverless** compute.

6. Leave the job page open and move to the next task.



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Job Validation
  </strong>
  <div style="color:#333;">

During development it would be beneficial to run and confirm the job works. 

For the purpose of this demonstration the job has been tested and validated.

  </div>
</div>



### B3. View the Job YAML Configuration

Complete the following steps to view the job configuration as code:

1. Go back to your job.

2. In the top-right corner of the job page, click the kebab menu (three vertical dots) near the **Run now** button.

3. Select **View as code**.
    - **View jobs as code** documentation:
    [AWS](https://docs.databricks.com/aws/en/jobs/automate#view-jobs-as-code) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/jobs/automate#view-jobs-as-code) |
    [GCP](https://docs.databricks.com/gcp/en/jobs/automate#view-jobs-as-code)
4. Notice that Databricks can generate the job configuration in multiple formats.

| Format | What You Can Do |
|---|---|
| **YAML** | - View the job as YAML configuration<br>- Click **Copy** to paste the configuration directly into Declarative Automation Bundle `.yaml` files<br>- Click **Edit** to modify the job configuration in YAML instead of using the UI |
| **Python** | - Choose between **Databricks SDK** or **Declarative Automation Bundles** format<br>- Click **Copy** to generate reusable Python code<br>- Use the **Databricks SDK** version to create jobs in notebooks or local development environments<br>- Use the **Bundles** version to define jobs in Python-based bundle configurations |
| **JSON** | - Click **Copy** to retrieve the full job configuration in JSON format<br>- Use the JSON with the Databricks CLI, Databricks SDKs, or the Databricks REST API to create, update, or retrieve jobs |


5. Copy the **YAML** configuration.
   
6. Select **Close**.

7. Leave the tab with your job open. We use this copied YAML configuration in a later section.

#### Checkpoint - Example YAML Configuration (your values will differ slightly)
```
resources:
  jobs:
    demo05_simple_dab_labuser123:
      name: demo05_simple_dab_labuser123
      tasks:
        - task_key: create_bronze_table
          notebook_task:
            notebook_path: /Workspace/Users/your_user/automated-deployment-with-declarative-automation-bundles-source/labs/Source/en_us/notebooks/Automated
              Deployment with Declarative Automation Bundles/05 - Deploying a Simple DAB/src/create_bronze_table
            source: WORKSPACE
        - task_key: create_silver_table
          depends_on:
            - task_key: create_bronze_table
          notebook_task:
            notebook_path: /Workspace/Users/your_user/automated-deployment-with-declarative-automation-bundles-source/labs/Source/en_us/notebooks/Automated
              Deployment with Declarative Automation Bundles/05 - Deploying a Simple DAB/src/create_silver_table
            source: WORKSPACE
      parameters:
        - name: display_target
          default: development
        - name: catalog_name
          default: labuser123_1_dev
```

## C. Deploying your Job Using Declarative Automation Bundles (DABs)

### C1. Run Databricks CLI Commands

1. Run the `databricks -v` command to view the version of the Databricks CLI. 

    Confirm that the cell returns version **v0.298.0**.

In [0]:
%sh
databricks -v

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    DATABRICKS CLI ERROR TROUBLESHOOTING:
  </strong>
  <div style="color:#333;">

  - If you encounter a Databricks CLI authentication error, it means the authentication was not successful. Confirm you ran the notebook using your **all purpose compute**.

  - If you encounter the error below, it means your **databricks.yml** file has syntax issues due to a modification. Even for non-DAB CLI commands, the **databricks.yml** file is still required, as it may contain important authentication details, such as the host and profile, which are utilized by the CLI commands.

![CLI Invalid YAML](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/databricks_cli_error_invalid_yaml.png)
  </div>
</div>


2. Use the `pwd` command to view the current working directory. 

    It should display that you are in the folder **05 - Deploying a Simple DAB**. 
      - The CLI is using the current directory of this notebook.

In [0]:
%sh
pwd

3. Use the `ls` command to view the available files in the current directory. 

    Confirm that you see the **databricks.yml** file.

In [0]:
%sh
ls


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Notes
  </strong>
  <div style="color:#333;">

A bundle must contain one (and only one) configuration file named **databricks.yml** at the root of the bundle project folder.

The **databricks.yml** file is the main configuration file that defines a bundle, but it can reference other configuration files, such as resource configuration files, in the include mapping.

A bundle configuration file must be in YAML format and must contain at least the top-level `bundle` mapping.
  </div>
</div>

### C2. Explore the Simple **databricks.yml** Bundle Configuration File

Now that we have confirmed we are in the working directory of the **databricks.yml** file, let's open the bundle configuration file in a new tab and explore the bundle configuration.

For the full list of bundle configuration keys, view the **Configuration reference** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/reference) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/reference) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/reference)

1. In the left workspace navigation, select the folder icon and confirm you are in the **05 - Deploying a Simple DAB** folder.

2. Right-click on the **databricks.yml** file and select *Open in a new tab*.

3. In the **databricks.yml** file, a configuration must contain only one top-level `bundle` mapping.
    - This `bundle` mapping must contain a `name` mapping that specifies a programmatic (or logical) name for the bundle.
        ```
        bundle:                   # Required
          name: demo05_bundle     # Required
        ```

4. The `resources` mapping (notice this is blank in the YAML) specifies:
    - Information about the Databricks resources used by the bundle.
    - This bundle configuration defines a job resource. We will add our specific job in the next section and review the configuration.

5. The `targets` mapping specifies:
    - One or more target environments in which to run a Databricks workflow.
    - Each target is a unique collection of artifacts, Databricks workspace settings, and Databricks job or pipeline details.
    - In this example, we have one target named `development` and it uses a simple configuration.

6. The `mode: development` mapping:
    - Defines this target as `development` mode.
    - Development mode implements a variety of behaviors. For example:
        - Prepends all resources that are not deployed as files or notebooks with the prefix **[dev ${workspace.current_user.short_name}]**
        - Tags each deployed job and pipeline with a `dev` Databricks tag.
    - For more behaviors, view the **Development mode** documentation:
    [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes#development-mode) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/deployment-modes#development-mode) |
    [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/deployment-modes#development-mode)

7. The `default: true` mapping specifies that:
    - This is the default target environment if multiple targets are available.
    - Setting the default to the **development** target helps avoid accidentally deploying to a production environment.

8. In the `workspace` mapping the following are specified:
    - `host` specifies the workspace to run this in. By default it will use the current workspace. We will leave this commented out.
    - `root_path` specifies where the files will be deployed.

### C3. Add Our Job Configuration to the **databricks.yml** File

After examining the bundle configuration in the **databricks.yml** file, let's go back to our job and copy the YAML configuration (if necessary).

1. In your **databricks.yml** file paste your job YAML configuration in the **resources** mapping with your specific job YAML configuration (under the RESOURCES comment).

After pasting your specific job configuration to your **databricks.yml** file, let's modify some of the paths to make them relative paths, add the notebook extensions, and give it an easy job key name.

2. Under `resources` > `jobs` you will see a key named `demo05_simple_dab_username`.
      - Replace that key with `demo05_simple_dab`.

   ```
   resources:
      jobs:
        demo05_simple_dab_labuser1234:    ## <--------MODIFY THIS VALUE HERE TO demo05_simple_dab
          name: demo05_simple_dab_labuser1234
   ```

3. For `task_key: create_bronze_table`:
      - Modify the `notebook_path` to: `./src/create_bronze_table.ipynb`.

4. For `task_key: create_silver_table`:
      - Modify the `notebook_path` to: `./src/create_silver_table.ipynb`.

5. Close the **databricks.yml** file.

#### Checkpoint - Example YAML Configuration (your values will differ slightly)
```
...
resources:
  jobs:
    demo05_simple_dab:
      name: demo05_simple_dab_labuser1234
      tasks:
        - task_key: create_bronze_table
          notebook_task:
            notebook_path: ./src/create_bronze_table.ipynb
            source: WORKSPACE
        - task_key: create_silver_table
          depends_on:
            - task_key: create_bronze_table
          notebook_task:
            notebook_path: ./src/create_silver_table.ipynb
            source: WORKSPACE
      parameters:
        - name: display_target
          default: development
        - name: catalog_name
          default: labuser1234_1_dev
...
```


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Notebook Format Information
  </strong>
  <div style="color:#333;">

Notebooks can be in a variety of extensions: `.ipynb`, `.sql`, `.py`. Always confirm the extension.

- In the top navigation bar, below the notebook name, select **File**.

- Scroll down and find the **Notebook format** option, then select it.

- Here, you should see the notebook format listed as **Source (.ipynb, .py, .sql, etc)**.
  </div>
</div>

### C4. Validate Your Bundle
Let's validate our **databricks.yml** bundle configuration file using the Databricks CLI.

1.  Run the cell and confirm the validation of the bundle was successful.

In [0]:
%sh
databricks bundle validate

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
     Troubleshooting
  </strong>
  <div style="color:#333;">
If you see the following error after validating your bundle, the format of your notebook could be incorrect.

`Error: notebook src/create_bronze_table.ipynb not found`. 

Check the format of your notebook and adjust accordingly. 

  </div>
</div>


2. Deploy the bundle using the Databricks CLI. Run the command below to deploy the bundle. 

    - The command `databricks bundle deploy -t development` specifies to deploy the bundle to the `development` environment. 
    - By default if we did not specify the target environment it would use the default target we specified earlier (development).

    **NOTE:** This will take about a minute to complete.

In [0]:
%sh
databricks bundle deploy -t development

### C5. View the Deployed Job

1. Let's view where the Databricks assets were deployed.

    a. In the main navigation bar, right-click on **Workspace** and select **Open in a New Tab**.

    b. Navigate to **Workspace > Users > your user name** > **.bundle** folder.

    c. Open **demo05_bundle** (the bundle name we specified in **databricks.yml**).

    d. Here, we can see that we deployed the **development** target. 

    - Within the **development** folder, there will be a variety of folders and files.

    e. Close the Workspace tab.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

Because you deployed the bundle **from within the Databricks workspace**, the deployment used a **source-linked deployment** for the **development** target.

In a source-linked deployment:
  - The source files are **not copied** into the target deployment folder
  - The deployment references the existing workspace files directly

You can confirm this by checking the following folder. It will be empty:

```text
.bundle/demo05_bundle/development/files
```
  </div>
</div>


2. Complete the following steps to explore the job we deployed with a DAB.

    a. In the left main navigation bar, right-click on **Jobs & Pipelines** and select *Open in a new tab*.

    b. Find your deployed job named **[dev username] demo05_simple_dab**.

      - By default, development mode prepends all resources that are not deployed as files or notebooks with the prefix `[dev ${workspace.current_user.short_name}]` and tags each deployed job and pipeline with a `dev` Databricks tag.

      - For other **Development mode** behaviors, view the documentation:
      [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes#development-mode) |
      [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/deployment-modes#development-mode) |
      [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/deployment-modes#development-mode)

    c. Select the job.

    d. Notice the note at the top of the job: **Connected to Declarative Automation Bundles**.
    
    e. Select the link **Learn more** and read the note.

    f. In the right navigation pane, scroll down to **Job parameters**. Notice the values of the job parameters:

      - `catalog_name` - your dev catalog
      - `display_target` - the value `development`

    g. Leave the job tab open.


### C6. Run the Job Using the Databricks CLI

1. Run the cell below to run the job from the **databricks.yml** file using the CLI command and confirm the job runs successfully.

      - `databricks bundle run -t development demo05_simple_dab` specifies to run this job in the development environment.

      - This job key can be found under the **resources** mapping in the **databricks.yml** file.

**Example (your actual job `name` will differ)**:
```
resources:
  jobs:
    demo05_simple_dab:    # <--- The job key. Your job key should be: demo05_simple_dab
      name: demo05_simple_dab_labuser1234   # <--- The job name (auto-generated, will differ)
```

In [0]:
%sh
databricks bundle run -t development demo05_simple_dab

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting
  </strong>
  <div style="color:#333;">
  
If the bundle run returns the following error: `Error: resource with key "demo05_simple_dab" not found`.

That means you did not modify the job key correctly. View your `resources` mapping and confirm the job key is `demo05_simple_dab`.

```yaml
resources:
  jobs:
    demo05_simple_dab:   # <--- This job key here
      name: demo05_simple_dab_labuser1234
      tasks:
```
  </div>
</div>


2. After the job was successfully run, navigate back to the job tab. 

    Notice that the cell above automatically ran the specified job using our development catalog that we specified within the job parameters in the **databricks.yml** file.

![Job Run 1](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/simple-dab/job-run-1.png)

3. Run the cell below and confirm the following tables were created from the job in our **labuser_UNIQUE_ID_1_dev** catalog:
    - **health_bronze_demo_05**
    - **health_silver_demo_05**

In [0]:
tables = spark.sql(f'''
SHOW TABLES IN {catalog_dev}.default
''')

tables.display()

### C7. Modify the **databricks.yml** and Redeploy the Job

1. Let's make a change to our bundle configuration in the **databricks.yml** file.

    a. (If not already opened) Right click on the **databricks.yml** file and select *Open in a new tab*.

    b. In the **resources** mapping modify the following:

    - The default value of the job parameter `display_target` to `development_updating_the_value_test`.

    c. Run the cell below to validate and deploy the new bundle.

    - Wait until the cell completes (about 1 minute).

In [0]:
%sh
databricks bundle validate
databricks bundle deploy -t development


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information - Must Redeploy if you update the databricks.yml configuration file
  </strong>
  <div style="color:#333;">

If you make a change to your configuration file you will have to redeploy the bundle. 

After you modify the **databricks.yml** file wait about 30 seconds for the auto save to save the file before redeploying.

  </div>
</div>


2. After the deployment completes, view the new deployed job by:
    - Navigating back to your job
    - Then view the **Job parameters** (if the page is already open, refresh the page). 

    Notice that the default value for the **display_target** parameter has been updated based on the change we made in the **databricks.yml** file.

![Job Run 2](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/simple-dab/job-run-2.png)

## D. Destroy the Deployed Job

1. Lastly since we are finished with this bundle, let's delete it using the `databricks bundle destroy` command.

    By default, you are prompted to confirm permanent deletion of the previously-deployed jobs, pipelines, and artifacts. To skip these prompts and perform automatic permanent deletion, add the `--auto-approve` option to the bundle destroy command.

In [0]:
%sh
databricks bundle destroy --auto-approve


<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Destroy Warning</strong>
  <div style="color:#333;">

Destroying a bundle permanently deletes a bundle's previously-deployed jobs, pipelines, and artifacts. This action cannot be undone.

For more information, view the **Destroy the bundle** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/work-tasks#step-6-destroy-the-bundle) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/work-tasks#step-6-destroy-the-bundle) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/work-tasks#step-6-destroy-the-bundle)

  </div>
</div>

## Conclusion

In this demonstration, you walked through the complete lifecycle of a Declarative Automation Bundle (DAB) for a simple Databricks job:

1. Generated a YAML job configuration from an existing job using **View as code**.
2. Pasted that configuration into the bundle's **databricks.yml** under the `resources.jobs` mapping.
3. Validated the bundle with `databricks bundle validate`.
4. Deployed the bundle to the `development` target with `databricks bundle deploy -t development`.
5. Ran the deployed job with `databricks bundle run -t development demo05_simple_dab`.
6. Modified a job parameter, redeployed, and verified the change in the UI.
7. Cleaned up by destroying the bundle with `databricks bundle destroy --auto-approve`.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>